In [1]:
import os
import numpy as np
import pandas as pd
import torch
import faiss

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)

from sentence_transformers import SentenceTransformer

In [2]:
PROJECT_ROOT = os.path.abspath("..")

CLASSIFIER_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "security_classifier",
    "best_model"
)

FAISS_PATH = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "faiss",
    "knowledge_base.index"
)

KB_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "knowledge_base.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "rag"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Paths configured.")

Paths configured.


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

LABEL_MAP = {
    0: "safe",
    1: "malicious",
    2: "phi",
    3: "jailbreak",
    4: "suspicious"
}

classifier_tokenizer = AutoTokenizer.from_pretrained(
    CLASSIFIER_PATH
)

classifier = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_PATH
)

classifier.to(device)
classifier.eval()

print("Security classifier loaded.")
print("Device:", device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Security classifier loaded.
Device: cpu


In [4]:
EMBEDDING_MODEL_NAME = (
    "pritamdeka/BioBERT-mnli-snli-scinsli-scitail-mednli-stsb"
)

In [7]:
# Corrected Hugging Face repository name
EMBEDDING_MODEL_NAME = (
    "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-sts"
)

# Use an available public BioBERT sentence-transformer model
EMBEDDING_MODEL_NAME = "pritamdeka/S-BioBert-snli-multinli-stsb"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded.")

print("Embedding model loaded.")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\raich\anaconda3\envs\llm-security\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raich\.cache\huggingface\hub\models--pritamdeka--S-BioBert-snli-multinli-stsb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.32k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Embedding model loaded.
Embedding model loaded.


In [8]:
faiss_index = faiss.read_index(
    FAISS_PATH
)

kb = pd.read_csv(
    KB_PATH
)

print("FAISS vectors:", faiss_index.ntotal)
print("Knowledge-base documents:", len(kb))

assert faiss_index.ntotal == len(kb)

print("✓ FAISS and KB are aligned.")

FAISS vectors: 15979
Knowledge-base documents: 15979
✓ FAISS and KB are aligned.


In [9]:
def classify_prompt(prompt):

    inputs = classifier_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = classifier(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    predicted_id = torch.argmax(
        probabilities
    ).item()

    confidence = probabilities[
        predicted_id
    ].item()

    return {
        "label_id": predicted_id,
        "label": LABEL_MAP[predicted_id],
        "confidence": confidence,
        "probabilities": {
            LABEL_MAP[i]: float(probabilities[i])
            for i in range(len(LABEL_MAP))
        }
    }

In [10]:
def retrieve_documents(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if idx == -1:
            continue

        document = kb.iloc[idx]

        results.append({
            "rank": rank,
            "document_index": int(idx),
            "similarity": float(score),
            "source_dataset": document[
                "source_dataset"
            ],
            "prompt": document["prompt"],
            "response": document["response"]
        })

    return pd.DataFrame(results)

In [11]:
def build_context(
    retrieval_results
):

    context_parts = []

    for _, row in retrieval_results.iterrows():

        context_parts.append(
            f"Reference {int(row['rank'])}:\n"
            f"Question: {row['prompt']}\n"
            f"Answer: {row['response']}"
        )

    return "\n\n".join(
        context_parts
    )

In [12]:
LLM_NAME = "google/flan-t5-base"

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME
)

llm_model.to(device)
llm_model.eval()

print("LLM loaded.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

c:\Users\raich\anaconda3\envs\llm-security\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raich\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

ValueError: Unrecognized configuration class <class 'transformers.models.t5.configuration_t5.T5Config'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of GPT2Config, AfmoeConfig, ApertusConfig, ArceeConfig, AriaTextConfig, AXK1Config, AXK2Config, BambaConfig, BartConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BloomConfig, BltConfig, CamembertConfig, CodeGenConfig, CohereConfig, Cohere2Config, Cohere2MoeConfig, CohereCompassTextConfig, CpmAntConfig, CTRLConfig, CwmConfig, Data2VecTextConfig, DbrxConfig, DeepseekV2Config, DeepseekV3Config, DeepseekV32Config, DeepseekV4Config, DiffLlamaConfig, DogeConfig, Dots1Config, ElectraConfig, Emu3Config, ErnieConfig, Ernie4_5Config, Ernie4_5_MoeConfig, Exaone4Config, ExaoneMoeConfig, FalconConfig, FalconH1Config, FalconMambaConfig, FlexOlmoConfig, FuyuConfig, GemmaConfig, Gemma2Config, Gemma3Config, Gemma3TextConfig, Gemm